**Universidad Internacional de La Rioja (UNIR) - Máster Universitario en Inteligencia Artificial - Razonamiento y Planificación Automática**

***
Datos del alumno (Nombre y Apellidos): Jose Manuel Pinillos Rubio

Fecha: 27 de noviembre de 2025
***

# <span style="font-size: 20pt; font-weight: bold; color: #0098cd;">Actividad: Búsqueda de rutas en empresa de paquetería</span>


## 1. Instalación de dependencias

In [60]:
!pip install simpleai flask pydot graphviz

## 2. Importación de librerías y módulos necesarios

In [61]:
from __future__ import print_function

import math
from simpleai.search.viewers import BaseViewer,ConsoleViewer,WebViewer
from simpleai.search import SearchProblem, astar, breadth_first, depth_first, uniform_cost

## 3. Implementación del entorno y configuración del experimento

Este bloque contiene todo el código necesario para definir el entorno del problema, implementar los algoritmos de búsqueda y gestionar la ejecución y visualización de los resultados.

### 3.1 - Definición de la clase del problema de búsqueda, funciones del entorno y heurísticas


In [62]:
# Definición de la clase del problema de búsqueda
class GameWalkPuzzle(SearchProblem):

    def __init__(self, board, costs, heuristic_number):
        self.board = board
        self.goal = (0, 0)
        self.costs = costs
        self.heuristic_number = heuristic_number
        for y in range(len(self.board)):
            for x in range(len(self.board[y])):
                if self.board[y][x].lower() == "t":
                    self.initial = (x, y)
                elif self.board[y][x].lower() == "p":
                    self.goal = (x, y)

        super(GameWalkPuzzle, self).__init__(initial_state=self.initial)


    # Implementación de funciones del problema de búsqueda
    def actions(self, state):
        actions = []
        for action in list(self.costs.keys()):
            newx, newy = self.result(state, action)
            if self.board[newy][newx] != "#":
                actions.append(action)
        return actions

    def result(self, state, action):
        x, y = state

        if action.count("up"):
            y -= 1
        if action.count("down"):
            y += 1
        if action.count("left"):
            x -= 1
        if action.count("right"):
            x += 1

        new_state = (x, y)
        return new_state

    def is_goal(self, state):
        return state == self.goal

    def cost(self, state, action, state2):
        return self.costs[action]


    # Definición de funciones heurísticas

    #Esta función heurística es la distancia entre el estado actual y el objetivo (único) identificado como self.goal
    def heuristic1(self, state):
        x, y = state
        gx, gy = self.goal
        return abs(x - gx) + abs(y - gy)

    def heuristic2(self, state):
        x, y = state
        gx, gy = self.goal
        return max(abs(x - gx),abs(y - gy))

    def heuristic3(self, state):
        x, y = state
        gx, gy = self.goal
        return 2*(abs(x - gx) + abs(y - gy))

    def heuristic(self,state):
      if self.heuristic_number == 1:
          return self.heuristic1(state)
      elif self.heuristic_number == 2:
          return self.heuristic2(state)
      elif self.heuristic_number == 3:
          return self.heuristic3(state)
      else:
        raise Exception("El número de la función heurística debe estar entre 1 y 3. Revise la inicialización del problema.")

### 3.2 - Cálculo de métricas del resultado de la búsqueda

In [63]:
def searchInfo (problem,result,use_viewer):
    if result is None:
        return "No se encontró solución.\n"
    def getTotalCost (problem,result):
        originState = problem.initial_state
        totalCost = 0
        for action,endingState in result.path():
            if action is not None:
                totalCost += problem.cost(originState,action,endingState)
                originState = endingState
        return totalCost


    res = "Total length of solution: {0}\n".format(len(result.path()))
    res += "Total cost of solution: {0}\n".format(getTotalCost(problem,result))

    if use_viewer:
        stats = [{'name': stat.replace('_', ' '), 'value': value}
                         for stat, value in list(use_viewer.stats.items())]

        for s in stats:
            res+= '{0}: {1}\n'.format(s['name'],s['value'])
    return res

### 3.3 - Visualización del resultado sobre el mapa

In [64]:
def resultado_experimento(problem,MAP,result,used_viewer):
    if result is None:
        print("No se encontró solución.")
        print(searchInfo(problem,result,used_viewer))
        return

    path = [x[1] for x in result.path()]

    for y in range(len(MAP)):
        for x in range(len(MAP[y])):
            if (x, y) == problem.initial:
                print("T", end='')
            elif (x, y) == problem.goal:
                print("P", end='')
            elif (x, y) in path:
                print("·", end='')
            else:
                print(MAP[y][x], end='')
        print()

    info=searchInfo(problem,result,used_viewer)
    print(info)

### 3.4 - Función principal de ejecución del experimento

In [65]:
def main(MAP_ASCII,COSTS,algorithms,heuristic_number=1):
    MAP = [list(x) for x in MAP_ASCII.split("\n") if x]

    for algorithm in algorithms:
      problem = GameWalkPuzzle(MAP,COSTS,heuristic_number)
      used_viewer=BaseViewer()
      # Probad también ConsoleViewer para depurar
      # No podréis usar WebViewer en Collab para ver los árboles

      # Mostramos tres experimentos
      print ("Experimento con algoritmo {}:".format(algorithm))

      result = algorithm(problem, graph_search=True,viewer=used_viewer)
      resultado_experimento(problem,MAP,result,used_viewer)

### 3. - Definición del mapa base del entorno

In [66]:
# Mapa para todos los casos

MAP_ASCII = """
#########
# P     #
# # ##  #
#    #  #
# ##T   #
#       #
#########
"""

## 4. Ejecución del caso 1: Comparación entre BFS y DFS

En esta sección se ejecutan los algoritmos de búsqueda no informada: **búsqueda en amplitud (*Breadth-First Search*)** y búsqueda en profundidad (*Depth-First Search*) **texto en negrita**, utilizando un entorno con costes uniformes para todas las direcciones. El objetivo es observar las diferencias de comportamiento entre ambas estrategias en términos de optimalidad, tiempo y memoria.

In [67]:
# Configuración y llamada para el caso 1
# Se ejecutan los algoritmos de búsqueda en amplitud y búsqueda en profundidad

COSTS = {
    "left": 1.0,
    "right": 1.0,
    "up": 1.0,
    "down": 1.0,
}

algorithms=(breadth_first,depth_first)
main (MAP_ASCII,COSTS,algorithms)

Experimento con algoritmo <function breadth_first at 0x781367a3ede0>:
#########
# P·    #
# #·##  #
#  ··#  #
# ##T   #
#       #
#########
Total length of solution: 6
Total cost of solution: 5.0
max fringe size: 7
visited nodes: 25
iterations: 25

Experimento con algoritmo <function depth_first at 0x781367a3dd00>:
#########
# P···· #
# # ##· #
#    #· #
# ##T · #
#   ··· #
#########
Total length of solution: 12
Total cost of solution: 11.0
max fringe size: 10
visited nodes: 22
iterations: 22



## 5. Ejecución del caso 2: Comparación entre BFS, UCS y A*

Aquí se utiliza el mismo mapa base, pero se introducen costes diferenciales por dirección de movimiento. Se comparan los algoritmos de **búsqueda en amplitud (BFS)**, **búsqueda de coste uniforme (UCS)** y **A***, con una heurística consistente. Este caso permite evaluar cómo se comportan algoritmos informados y no informados ante costes variables.

In [68]:
# Configuración y llamada para el caso 2
# Se utiliza el mismo mapa pero se varían los costes

COSTS = {
    "left": 3.0,
    "right": 1.0,
    "up": 1.0,
    "down": 3.0,
}

algorithms=(breadth_first,uniform_cost,astar)
main (MAP_ASCII,COSTS,algorithms)

Experimento con algoritmo <function breadth_first at 0x781367a3ede0>:
#########
# P·    #
# #·##  #
#  ··#  #
# ##T   #
#       #
#########
Total length of solution: 6
Total cost of solution: 9.0
max fringe size: 7
visited nodes: 25
iterations: 25

Experimento con algoritmo <function uniform_cost at 0x781367a3d080>:
#########
# P·    #
# #·##  #
#  ··#  #
# ##T   #
#       #
#########
Total length of solution: 6
Total cost of solution: 9.0
max fringe size: 7
visited nodes: 23
iterations: 23

Experimento con algoritmo <function astar at 0x781367bffec0>:
#########
# P·    #
# #·##  #
#  ··#  #
# ##T   #
#       #
#########
Total length of solution: 6
Total cost of solution: 9.0
max fringe size: 9
visited nodes: 11
iterations: 11



### 5.1 - Evaluación del comportamiento con costes negativos

En esta subsección se introducen **costes negativos en algunas direcciones** con el objetivo de analizar cómo afectan al comportamiento de los algoritmos de búsqueda cuando se rompe una de las condiciones que garantizan su correcto funcionamiento.

Tal como se explicó en el Caso 2, algoritmos como UCS pueden dejar de ser óptimos e incluso entrar en bucles si existen ciclos con coste negativo, mientras que A* pierde sus garantías de optimalidad si la heurística no es consistente.

Esta prueba permite observar de manera práctica cómo varían los resultados cuando se favorecen ciertas transiciones mediante un coste negativo y comprobar si los algoritmos siguen siendo capaces de encontrar soluciones válidas o presentan desviaciones respecto a los casos anteriores.

In [69]:
COSTS = {
    "left": 3.0,
    "right": -4.0,
    "up": 1.0,
    "down": 3.0,
}

algorithms=(breadth_first,uniform_cost,astar)
main (MAP_ASCII,COSTS,algorithms)

Experimento con algoritmo <function breadth_first at 0x781367a3ede0>:
#########
# P·    #
# #·##  #
#  ··#  #
# ##T   #
#       #
#########
Total length of solution: 6
Total cost of solution: 9.0
max fringe size: 7
visited nodes: 25
iterations: 25

Experimento con algoritmo <function uniform_cost at 0x781367a3d080>:
#########
# P·····#
# # ## ·#
#    # ·#
# ##T···#
#       #
#########
Total length of solution: 12
Total cost of solution: 6.0
max fringe size: 8
visited nodes: 23
iterations: 23

Experimento con algoritmo <function astar at 0x781367bffec0>:
#########
# P·····#
# # ## ·#
#    # ·#
# ##T···#
#       #
#########
Total length of solution: 12
Total cost of solution: 6.0
max fringe size: 8
visited nodes: 18
iterations: 18



## 6. Ejecución del caso 3: Comparación de heurísticas en A*

En este caso se analiza exclusivamente el comportamiento del algoritmo A* utilizando tres funciones heurísticas diferentes:

- heuristic1: distancia de Manhattan
  ```
  return abs(x - gx) + abs(y - gy)
  ```

- heuristic2: distancia de Chebyshev
  ```
  return max(abs(x - gx), abs(y - gy))
  ```

- heuristic3: distancia de Manhattan multiplicada por 2
  ```
  return 2 * (abs(x - gx) + abs(y - gy))
  ```

Todas las pruebas se realizan sobre el mismo entorno y con los mismos costes del caso 2.

En primer lugar, se ejecuta A* con cada una de las tres heurísticas sobre el mapa base. Posteriormente, se incluyen dos variaciones para comprobar si el algoritmo sigue encontrando soluciones óptimas en otros contextos:

- En la primera variación, se mantiene el mismo entorno, pero se modifica la posición de inicio, trasladándola a una zona distinta del mapa.
- En la segunda variación, se emplea un mapa completamente diferente, más complejo y con múltiples rutas posibles.

Estas pruebas adicionales permiten analizar si el comportamiento observado en el entorno base se mantiene al introducir cambios estructurales, especialmente cuando se utiliza una heurística no admisible como la número 3. Así, se puede contrastar su impacto real en la optimalidad y la eficiencia de A*.

In [70]:
# Configuración y llamada para el caso 3
# Se utiliza el mismo mapa y se usan diferentes heurísticas


COSTS = {
    "left": 0.0,
    "right": 3.0,
    "up": 1.0,
    "down": 1.0,
}

algorithms=(astar,)
main (MAP_ASCII,COSTS,algorithms,1)
main (MAP_ASCII,COSTS,algorithms,2)
main (MAP_ASCII,COSTS,algorithms,3)

Experimento con algoritmo <function astar at 0x781367bffec0>:
#########
# P·    #
# #·##  #
#  ··#  #
# ##T   #
#       #
#########
Total length of solution: 6
Total cost of solution: 3.0
max fringe size: 6
visited nodes: 8
iterations: 8

Experimento con algoritmo <function astar at 0x781367bffec0>:
#########
# P·    #
# #·##  #
#  ··#  #
# ##T   #
#       #
#########
Total length of solution: 6
Total cost of solution: 3.0
max fringe size: 6
visited nodes: 9
iterations: 9

Experimento con algoritmo <function astar at 0x781367bffec0>:
#########
# P·    #
# #·##  #
#  ··#  #
# ##T   #
#       #
#########
Total length of solution: 6
Total cost of solution: 3.0
max fringe size: 5
visited nodes: 7
iterations: 7



### 6.1 – Prueba con cambio de punto de partida

#### 6.1.1 – Nuevo mapa: modificación de la posición de inicio

En esta variante se mantiene la estructura original del mapa, pero se modifica la ubicación del punto de partida para comprobar si las tres heurísticas siguen produciendo resultados óptimos bajo una nueva configuración inicial.

In [71]:
MAP_ASCII = """
#########
# P     #
# # ##  #
#    #  #
# ##    #
#   T   #
#########
"""

#### 6.1.2 – Ejecución con las tres heurísticas

Se vuelve a ejecutar el algoritmo A* con cada una de las funciones heurísticas ya implementadas (Manhattan, Chebyshev y Manhattan×2) para evaluar si los resultados se mantienen estables y óptimos.

In [72]:
COSTS = {
    "left": 3.0,
    "right": 1.0,
    "up": 1.0,
    "down": 3.0,
}

algorithms=(astar,)
main (MAP_ASCII,COSTS,algorithms,1)
main (MAP_ASCII,COSTS,algorithms,2)
main (MAP_ASCII,COSTS,algorithms,3)

Experimento con algoritmo <function astar at 0x781367bffec0>:
#########
# P·    #
# #·##  #
#  ··#  #
# ##·   #
#   T   #
#########
Total length of solution: 7
Total cost of solution: 10.0
max fringe size: 7
visited nodes: 14
iterations: 14

Experimento con algoritmo <function astar at 0x781367bffec0>:
#########
# P·    #
# #·##  #
#  ··#  #
# ##·   #
#   T   #
#########
Total length of solution: 7
Total cost of solution: 10.0
max fringe size: 7
visited nodes: 19
iterations: 19

Experimento con algoritmo <function astar at 0x781367bffec0>:
#########
# P·    #
# #·##  #
#  ··#  #
# ##·   #
#   T   #
#########
Total length of solution: 7
Total cost of solution: 10.0
max fringe size: 6
visited nodes: 7
iterations: 7



#### 6.1.3 – Ejecución modificando los costes de las acciones

Se repite la ejecución del algoritmo A* utilizando las tres funciones heurísticas, esta vez bajo un entorno con costes de movimiento modificados. El objetivo es comprobar si los resultados óptimos se mantienen o si la sobreestimación de la heurística 3 provoca pérdida de optimalidad.

In [73]:
COSTS = {
    "left": 0.0,
    "right": 3.0,
    "up": 1.0,
    "down": 1.0,
}

algorithms=(astar,)
main (MAP_ASCII,COSTS,algorithms,1)
main (MAP_ASCII,COSTS,algorithms,2)
main (MAP_ASCII,COSTS,algorithms,3)

Experimento con algoritmo <function astar at 0x781367bffec0>:
#########
# P·    #
# #·##  #
#  ··#  #
# ##·   #
#   T   #
#########
Total length of solution: 7
Total cost of solution: 4.0
max fringe size: 5
visited nodes: 15
iterations: 15

Experimento con algoritmo <function astar at 0x781367bffec0>:
#########
# P·    #
# #·##  #
#  ··#  #
# ##·   #
#   T   #
#########
Total length of solution: 7
Total cost of solution: 4.0
max fringe size: 5
visited nodes: 14
iterations: 14

Experimento con algoritmo <function astar at 0x781367bffec0>:
#########
#·P     #
#·# ##  #
#·   #  #
#·##    #
#···T   #
#########
Total length of solution: 9
Total cost of solution: 7.0
max fringe size: 4
visited nodes: 9
iterations: 9



### 6.2 – Prueba con entorno más complejo

#### 6.2.1 – Nuevo mapa: variación estructural del entorno

Se introduce un entorno mucho más complejo, con múltiples rutas posibles, para comprobar si las heurísticas mantienen la optimalidad del resultado y cómo varía la eficiencia en un espacio más exigente.

In [74]:
MAP_ASCII = """
####################
#P            #    #
# ## # ###### # ## #
#             #    #
####### ## # # #####
#        # # #     #
# ###### # # ##### #
# #    # # #      ##
##### ###########  #
#    #           # #
# ## # ######### # #
# ## #         # # #
#    ####### # #  T#
####################
"""

#### 6.2.2 – Ejecución con las tres heurísticas

Se repite la ejecución del algoritmo A* con las tres funciones heurísticas para observar posibles diferencias de comportamiento, tanto en la solución como en la eficiencia (tiempo y memoria).

In [75]:
COSTS = {
    "left": 3.0,
    "right": 1.0,
    "up": 1.0,
    "down": 3.0,
}

algorithms=(astar,)
main (MAP_ASCII,COSTS,algorithms,1)
main (MAP_ASCII,COSTS,algorithms,2)
main (MAP_ASCII,COSTS,algorithms,3)

Experimento con algoritmo <function astar at 0x781367bffec0>:
####################
#P·····       #    #
# ## #·###### # ## #
#     ······· #    #
####### ## #·# #####
#        # #·#     #
# ###### # #·##### #
# #    # # #······##
##### ###########··#
#    #           #·#
# ## # ######### #·#
# ## #         # #·#
#    ####### # #  T#
####################
Total length of solution: 29
Total cost of solution: 62.0
max fringe size: 6
visited nodes: 71
iterations: 71

Experimento con algoritmo <function astar at 0x781367bffec0>:
####################
#P·····       #    #
# ## #·###### # ## #
#     ······· #    #
####### ## #·# #####
#        # #·#     #
# ###### # #·##### #
# #    # # #······##
##### ###########··#
#    #           #·#
# ## # ######### #·#
# ## #         # #·#
#    ####### # #  T#
####################
Total length of solution: 29
Total cost of solution: 62.0
max fringe size: 8
visited nodes: 79
iterations: 79

Experimento con algoritmo <function astar at 0x781367bffec0>:
####

### 6.3 – Prueba con modificación de la heurística 3

#### 6.3.1 - Redefinición de la clase del problema de búsquedapara modificar la heurística 3

In [76]:
# Definición de la clase del problema de búsqueda
class GameWalkPuzzle(SearchProblem):

    def __init__(self, board, costs, heuristic_number):
        self.board = board
        self.goal = (0, 0)
        self.costs = costs
        self.heuristic_number = heuristic_number
        for y in range(len(self.board)):
            for x in range(len(self.board[y])):
                if self.board[y][x].lower() == "t":
                    self.initial = (x, y)
                elif self.board[y][x].lower() == "p":
                    self.goal = (x, y)

        super(GameWalkPuzzle, self).__init__(initial_state=self.initial)


    # Implementación de funciones del problema de búsqueda
    def actions(self, state):
        actions = []
        for action in list(self.costs.keys()):
            newx, newy = self.result(state, action)
            if self.board[newy][newx] != "#":
                actions.append(action)
        return actions

    def result(self, state, action):
        x, y = state

        if action.count("up"):
            y -= 1
        if action.count("down"):
            y += 1
        if action.count("left"):
            x -= 1
        if action.count("right"):
            x += 1

        new_state = (x, y)
        return new_state

    def is_goal(self, state):
        return state == self.goal

    def cost(self, state, action, state2):
        return self.costs[action]


    # Definición de funciones heurísticas

    #Esta función heurística es la distancia entre el estado actual y el objetivo (único) identificado como self.goal
    def heuristic1(self, state):
        x, y = state
        gx, gy = self.goal
        return abs(x - gx) + abs(y - gy)

    def heuristic2(self, state):
        x, y = state
        gx, gy = self.goal
        return max(abs(x - gx),abs(y - gy))

    def heuristic3(self, state):
        x, y = state
        gx, gy = self.goal
        return 4*(abs(x - gx) + abs(y - gy))

    def heuristic(self,state):
      if self.heuristic_number == 1:
          return self.heuristic1(state)
      elif self.heuristic_number == 2:
          return self.heuristic2(state)
      elif self.heuristic_number == 3:
          return self.heuristic3(state)
      else:
        raise Exception("El número de la función heurística debe estar entre 1 y 3. Revise la inicialización del problema.")

#### 6.3.2 – Nuevo mapa: variación estructural del entorno

In [77]:
MAP_ASCII = """
#########################
#P          ##########  #
#           #           #
#           #           #
#           #           #
#           #           #
#           #           #
#           #           #
#           #           #
#           #           #
#           #           #
#           #           #
#           #           #
#                      T#
#########################
"""

#### 6.3.3 – Ejecución con las tres heurísticas

Se repite la ejecución del algoritmo A* con las tres funciones heurísticas para observar posibles diferencias de comportamiento, tanto en la solución como en la eficiencia (tiempo y memoria).

In [78]:
COSTS = {
    "left": 3.0,
    "right": 1.0,
    "up": 1.0,
    "down": 3.0,
}

algorithms=(astar,)
main (MAP_ASCII,COSTS,algorithms,1)
main (MAP_ASCII,COSTS,algorithms,2)
main (MAP_ASCII,COSTS,algorithms,3)

Experimento con algoritmo <function astar at 0x781367bffec0>:
#########################
#P··········##########  #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ············T#
#########################
Total length of solution: 35
Total cost of solution: 78.0
max fringe size: 13
visited nodes: 267
iterations: 267

Experimento con algoritmo <function astar at 0x781367bffec0>:
#########################
#P··········##########  #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ·#           #
#          ············T#
########